In [8]:
import pandas as pd

products = pd.read_csv("../data/books.csv")
products = products.drop(["title", "isbn"], axis=1)
products.head()

,id,author,parent_genre,genre,sub_genre,price,pages,publisher,year,description
0,0,Delia Owens,Fiction,"Crime, Thriller & Mystery",Thrillers and Suspense,3.56,422,Penguin Books,1954,Imprescindible para cualquier amante de la lec...
1,1,Paula Hawkins,Fiction,"Crime, Thriller & Mystery",Thrillers and Suspense,3.67,380,Random House,1967,"Narración brillante que combina suspense, emoc..."
2,2,Alex Michaelides,Fiction,"Crime, Thriller & Mystery",Thrillers and Suspense,2.98,272,Macmillan Publishers,1968,"Escrita con una prosa elegante, esta obra es u..."
3,3,Alex Michaelides,Lifestyle & Leisure,"Arts, Film & Photography",Theory & Criticism,2.14,284,Macmillan Publishers,2016,Un viaje apasionante a través de ideas que tra...
4,4,Alex Michaelides,Fiction,Literature & Fiction,"Crime, Thriller & Mystery",2.98,703,Penguin Books,1986,Un viaje apasionante a través de ideas que tra...


In [9]:
import os
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.metrics.pairwise import cosine_similarity


class ItemRecommendationEngine():

    def preprocess(
        self,
        df: pd.DataFrame,
    ) -> pd.DataFrame:

        pipeline = ColumnTransformer([
            (
                "cat",
                OneHotEncoder(
                    sparse_output=False,
                    handle_unknown="ignore",
                ),
                df.select_dtypes(
                    include=["str", "object", "category"],
                ).columns.tolist()
            ),
            (
                "num",
                MinMaxScaler(),
                df.select_dtypes(
                    include="number",
                ).columns.tolist(),
            ),
        ])

        return pd.DataFrame(
            pipeline.fit_transform(df),
            index=df.index,
        )

    def train(
        self,
        data: pd.DataFrame,
    ):
        features = self.preprocess(data.drop(columns="id"))

        sim = pd.DataFrame(
            cosine_similarity(features.values),
            index=data.set_index([i for i in data.columns]).index,
            columns=data["id"].values,
        )
        self.similarity_df = sim

        return self

    def predict(
        self,
        sample: pd.DataFrame,
        limit=6,
    ):
        if self.similarity_df is None:
            return None

        product_id = sample["id"].item()

        if product_id not in self.similarity_df.index:
            return None

        scores = (
            self.similarity_df[product_id]
            .drop(product_id)
            .nlargest(limit)
            .to_frame("similarity")
        )

        return scores

    def save_model(self, path: str) -> None:
        dir_name = os.path.dirname(path)
        if dir_name:
            os.makedirs(dir_name, exist_ok=True)
        self.similarity_df.to_json(path, orient="records", indent=4)

    @classmethod
    def load_model(cls, path: str) -> "ItemRecommendationEngine":
        df = pd.read_json(path, orient="records")
        df.index = df.index.astype(int)
        df.columns = df.columns.astype(int)
        engine = cls()
        engine.similarity_df = df
        return engine

# Train

In [10]:
engine = ItemRecommendationEngine().train(products)

In [11]:
engine.save_model(
    "../addons/bookstore_recommendation/static/models/item_similarity.json",
)

# Predict

In [12]:
# engine = ItemRecommendationEngine().load_model(
#     "../addons/bookstore_recommendation/static/models/item_similarity.json",
# )

# engine.similarity_df.head()

In [13]:
harry_potter = products.iloc[[75]]
harry_potter

,id,author,parent_genre,genre,sub_genre,price,pages,publisher,year,description
75,75,J.K. Rowling,Fiction,"Fantasy, Horror & Science Fiction",Fantasy,10.99,111,Random House,2014,Un relato profundo y emotivo que explora la co...


In [14]:
data = engine.predict(harry_potter)
data

,,,,,,,,,,similarity
id,author,parent_genre,genre,sub_genre,price,pages,publisher,year,description,
191,J.K. Rowling,Fiction,"Fantasy, Horror & Science Fiction",Fantasy,5.96,491,Random House,2000,"Narración brillante que combina suspense, emoción y reflexión filosófica.",0.839228
113,J.K. Rowling,Fiction,"Fantasy, Horror & Science Fiction",Fantasy,2.73,716,Oxford University Press,1990,Un relato profundo y emotivo que explora la condición humana con maestría literaria.,0.817396
295,Holly Black,Fiction,"Fantasy, Horror & Science Fiction",Fantasy,3.42,277,Penguin Books,2009,Un relato profundo y emotivo que explora la condición humana con maestría literaria.,0.699393
343,J.K. Rowling,Fiction,"Fantasy, Horror & Science Fiction",Fantasy,30.64,227,Oxford University Press,2007,"Narración brillante que combina suspense, emoción y reflexión filosófica.",0.695801
195,J.K. Rowling,Fiction,"Fantasy, Horror & Science Fiction",Fantasy,17.59,393,Simon & Schuster,1991,"Escrita con una prosa elegante, esta obra es un referente en la literatura contemporánea.",0.691048
215,J.K. Rowling,Fiction,"Fantasy, Horror & Science Fiction",Fantasy,6.72,613,Simon & Schuster,2003,"Un clásico moderno que mezcla aventura, drama y crítica social de forma magistral.",0.690301
